# Notebook 02 — Embeddings (Voyage + OpenAI + Qwen3-8B + all-mpnet-base-v2)

**Project:** Study 1 — Chatbot Agreeableness (DGK Lab)  
**Pipeline stage:** 2 of 8 (NLP Colab notebooks) + 1 integration Rmd  
**Runtime:** ~30-45 minutes (Qwen3-8B on A100 dominates; mpnet adds ~1 min)  
**API costs:** ~$0.03–0.05 total (essentially free)

## Purpose

Generate sentence/passage embeddings across four different model families for every text unit in the corpus:

1. **Voyage AI** — `voyage-3-large` (1024-dim, primary; closed-API, retrieval-optimized encoder)
2. **OpenAI** — `text-embedding-3-large` (3072-dim, secondary; closed-API, Matryoshka transformer)
3. **Alibaba/Qwen** — `Qwen3-Embedding-8B` (4096-dim, tertiary; open-weights, **decoder-based** with last-token pooling)
4. **sentence-transformers** — `all-mpnet-base-v2` (768-dim, **preregistered baseline**; open-weights, contrastive sentence-similarity)

Four models from four different lineages lets us test robustness of any downstream finding (BERTopic clusters, classifier accuracy, alignment trajectories) across embedding spaces. If results converge, that is convergent evidence. If they diverge, that is also informative — and we can identify which architectural assumptions matter for which questions.

**Why mpnet is here:** `all-mpnet-base-v2` was preregistered as the embedding model for this study (see prereg §10.1). We added Voyage / OpenAI / Qwen3 as current-generation upgrades. We retain mpnet as the preregistered baseline so the manuscript can report: *"Substantive findings replicate across the preregistered model and three current-generation models."* mpnet truncates inputs at 384 tokens (a real-world property of the preregistered model); we document the truncation rate in the diagnostic.

## Text units embedded (N=130 sample)

| Unit | Count | Use |
|---|---|---|
| User turns | 2,464 | Per-turn alignment, classifier features |
| Bot turns | 2,555 | Alignment partner |
| OE responses (non-blank) | 662 | Topic modeling, clustering |
| Full conversations (concatenated) | 130 | Conversation-level classifier |
| OE portfolios (4 primary OEs concatenated) | 129 | Participant-level clustering (1 participant has all-blank OEs) |
| **Total per model** | **5,940** | |

(Unique text hashes ≈ 5,648 since some short user turns repeat verbatim across participants.)

## Safety features

- **Content-hash caching.** Every embedding cached to disk by SHA256(text + model). Re-running reads from cache.
- **Cost ceiling.** $5 hard ceiling per API provider; aborts if exceeded.
- **Pilot pass.** Embeds first 10 items per model, prints diagnostics, waits for GO before full run.
- **Batch APIs.** Voyage and OpenAI batched (128 items per call) to minimize round trips.
- **Idempotent.** Crashing halfway through means restarting picks up where it left off.

## API keys required

Set these as Colab Secrets (left sidebar > key icon > add):
- `CharacterAI_Voyage`
- `CharacterAI_OpenAI`

Qwen3-Embedding-8B and all-mpnet-base-v2 have no API keys; they download weights from HuggingFace (~15GB and ~420MB respectively).

## Outputs

- `data/derived/embeddings/voyage_3_large.parquet`
- `data/derived/embeddings/openai_3_large.parquet`
- `data/derived/embeddings/qwen3_8b.parquet`
- `data/derived/embeddings/all_mpnet_base_v2.parquet`
- `data/derived/embeddings/_cache/` — content-hashed cache (kept across runs)
- `data/derived/embeddings/embedding_manifest.json` — what was embedded, when, costs

## GPU requirement

**This notebook requires a GPU runtime.** In Colab: Runtime > Change runtime type > A100 GPU (or L4/T4 if A100 unavailable). Qwen3-8B runs in bf16 (~16GB VRAM); A100 is comfortable, L4 is workable, T4 will be tight. mpnet is small and runs in <1GB VRAM (or on CPU in ~1 min).


## 1. Environment setup

In [ ]:
import sys, os, json, time, hashlib
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

print(f'Python: {sys.version.split()[0]}')
print(f'pandas: {pd.__version__}')
print(f'numpy:  {np.__version__}')
print(f'Run timestamp: {datetime.now().isoformat(timespec="seconds")}')

# GPU check
try:
    import torch
    if torch.cuda.is_available():
        print(f'\n[GPU] Available: {torch.cuda.get_device_name(0)}')
        print(f'[GPU] VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    else:
        print('\n[WARNING] No GPU detected. Qwen3-8B will be very slow on CPU.')
        print('          Switch to Runtime > Change runtime type > A100 GPU')
except ImportError:
    print('\n[INFO] torch not yet imported; will install with model loading.')

## 2. Project paths and configuration

In [ ]:
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
                        'DGK Lab - AI Training Data, CharacterAI, Tort Liability/'
                        'Analysis/Study 1')
else:
    PROJECT_ROOT = Path.cwd().parent if (Path.cwd().name == 'NLP') else Path.cwd()

DATA_DERIVED   = PROJECT_ROOT / 'data' / 'derived'
EMB_DIR        = DATA_DERIVED / 'embeddings'
CACHE_DIR      = EMB_DIR / '_cache'
OUTPUTS_NLP    = PROJECT_ROOT / 'outputs' / 'NLP'

EMB_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
(CACHE_DIR / 'voyage').mkdir(exist_ok=True)
(CACHE_DIR / 'openai').mkdir(exist_ok=True)
(CACHE_DIR / 'qwen3').mkdir(exist_ok=True)
(CACHE_DIR / 'mpnet').mkdir(exist_ok=True)

# Cost ceilings (USD) per provider — abort if cumulative cost exceeds this
COST_CEILING_VOYAGE = 5.00
COST_CEILING_OPENAI = 5.00

# Pricing (per 1M tokens, as of late 2025; update if pricing has changed)
VOYAGE_PRICE_PER_M = 0.18   # voyage-3-large
OPENAI_PRICE_PER_M = 0.13   # text-embedding-3-large

# Batch size for API calls
BATCH_SIZE_API = 128
BATCH_SIZE_QWEN = 8    # Smaller because of GPU memory
BATCH_SIZE_MPNET = 64  # mpnet is small, can batch more aggressively

# Required input parquets from Notebook 01
PATH_TURNS    = DATA_DERIVED / 'chat_turns_long.parquet'
PATH_OE       = DATA_DERIVED / 'open_ends_long.parquet'
PATH_META     = DATA_DERIVED / 'participant_metadata.parquet'

for p in [PATH_TURNS, PATH_OE, PATH_META]:
    if not p.exists():
        raise FileNotFoundError(f'Missing input from Notebook 01: {p}')

print(f'Project root:  {PROJECT_ROOT}')
print(f'Embeddings:    {EMB_DIR}')
print(f'Cache:         {CACHE_DIR}')
print(f'Cost ceilings: Voyage=${COST_CEILING_VOYAGE}, OpenAI=${COST_CEILING_OPENAI}')


## 3. API key loading

Read keys from Colab Secrets. If running locally, set them as environment variables before launching Jupyter.

In [ ]:
def get_secret(name: str) -> Optional[str]:
    """Try Colab Secrets first, then environment variable."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception as e:
            print(f'[WARN] Could not read Colab Secret {name}: {e}')
    return os.environ.get(name)

CharacterAI_Voyage = get_secret('CharacterAI_Voyage')
CharacterAI_OpenAI = get_secret('CharacterAI_OpenAI')

for name, key in [('CharacterAI_Voyage', CharacterAI_Voyage), ('CharacterAI_OpenAI', CharacterAI_OpenAI)]:
    if key is None:
        print(f'[ERROR] {name} not found. Set it in Colab Secrets (left sidebar > key icon).')
    else:
        print(f'[OK] {name} loaded (length={len(key)})')

assert CharacterAI_Voyage is not None, 'Voyage API key required'
assert CharacterAI_OpenAI is not None, 'OpenAI API key required'

## 4. Build the master embedding job table

One row per (text unit, model) pair. This is the work queue. We build it once, then process it in three passes (one per model). Each row gets a content hash that determines its cache file path.

In [ ]:
turns = pd.read_parquet(PATH_TURNS)
oes   = pd.read_parquet(PATH_OE)
meta  = pd.read_parquet(PATH_META)

print(f'Loaded: {len(turns):>5} turns, {len(oes):>4} OE responses, {len(meta):>3} participants')

In [ ]:
def build_text_units() -> pd.DataFrame:
    """Construct the master list of texts to embed.

    Each row has: unit_type, sona_id, unit_id, text, condition_label, plus any
    other metadata needed downstream (turn_idx, oe_field, etc.).
    """
    units = []

    # User turns
    user_turns = turns[turns['role'] == 'user'].copy()
    for _, row in user_turns.iterrows():
        text = (row['content'] or '').strip()
        if not text:
            continue
        units.append({
            'unit_type': 'user_turn',
            'sona_id': row['sona_id'],
            'unit_id': f"{row['sona_id']}__user_t{int(row['user_turn_idx'])}",
            'turn_idx': row['turn_idx'],
            'role_turn_idx': int(row['user_turn_idx']),
            'condition_label': row['condition_label'],
            'text': text,
            'n_chars': len(text),
        })

    # Bot turns
    bot_turns = turns[turns['role'] == 'assistant'].copy()
    for _, row in bot_turns.iterrows():
        text = (row['content'] or '').strip()
        if not text:
            continue
        units.append({
            'unit_type': 'bot_turn',
            'sona_id': row['sona_id'],
            'unit_id': f"{row['sona_id']}__bot_t{int(row['bot_turn_idx'])}",
            'turn_idx': row['turn_idx'],
            'role_turn_idx': int(row['bot_turn_idx']),
            'condition_label': row['condition_label'],
            'text': text,
            'n_chars': len(text),
        })

    # Open-end responses (non-blank only)
    oe_nonblank = oes[~oes['is_blank']].copy()
    for _, row in oe_nonblank.iterrows():
        text = row['response'].strip()
        if not text:
            continue
        units.append({
            'unit_type': 'oe_response',
            'sona_id': row['sona_id'],
            'unit_id': f"{row['sona_id']}__{row['oe_field']}",
            'oe_field': row['oe_field'],
            'is_primary': row['is_primary'],
            'condition_label': row['condition_label'],
            'text': text,
            'n_chars': len(text),
        })

    # Full conversation (concatenated all turns)
    for sona_id, grp in turns.sort_values(['sona_id', 'turn_idx']).groupby('sona_id'):
        cond = grp['condition_label'].iloc[0]
        # Format: "User: ... Bot: ... User: ..."
        parts = []
        for _, t in grp.iterrows():
            tag = 'User' if t['role'] == 'user' else 'Bot'
            parts.append(f"{tag}: {t['content']}")
        text = '\n'.join(parts).strip()
        if text:
            units.append({
                'unit_type': 'full_conversation',
                'sona_id': sona_id,
                'unit_id': f"{sona_id}__conversation",
                'condition_label': cond,
                'text': text,
                'n_chars': len(text),
            })

    # OE portfolio (concatenated 4 primary OEs)
    primary_oes = oes[oes['is_primary'] & ~oes['is_blank']].copy()
    for sona_id, grp in primary_oes.sort_values(['sona_id', 'oe_order']).groupby('sona_id'):
        cond = grp['condition_label'].iloc[0]
        parts = []
        for _, r in grp.iterrows():
            parts.append(f"[{r['oe_field']}] {r['response']}")
        text = '\n\n'.join(parts).strip()
        if text:
            units.append({
                'unit_type': 'oe_portfolio',
                'sona_id': sona_id,
                'unit_id': f"{sona_id}__oe_portfolio",
                'condition_label': cond,
                'text': text,
                'n_chars': len(text),
            })

    df = pd.DataFrame(units)
    return df

units_df = build_text_units()
print(f'Built text-unit table: {len(units_df):,} rows')
print(f'\nUnit-type counts:')
print(units_df['unit_type'].value_counts())
print(f'\nCharacter-length descriptives by unit_type:')
print(units_df.groupby('unit_type')['n_chars'].describe()[['count','mean','50%','max']].round(0))

## 5. Content hashing and cache helpers

Every (text, model) pair gets a unique cache key. We hash the text content (not just the unit_id) so that if a text is identical to another, we only embed it once per model.

In [ ]:
def hash_text(text: str) -> str:
    """SHA256 of UTF-8-encoded text. First 16 chars used as cache filename."""
    return hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]

def cache_path(model_dir: Path, text_hash: str) -> Path:
    return model_dir / f'{text_hash}.npy'

def cache_load(model_dir: Path, text_hash: str) -> Optional[np.ndarray]:
    p = cache_path(model_dir, text_hash)
    if p.exists():
        try:
            return np.load(p)
        except Exception as e:
            print(f'[WARN] Cache file corrupted, re-embedding: {p.name} ({e})')
            p.unlink()
            return None
    return None

def cache_save(model_dir: Path, text_hash: str, vec: np.ndarray) -> None:
    np.save(cache_path(model_dir, text_hash), vec.astype(np.float32))

# Add hashes to units_df
units_df['text_hash'] = units_df['text'].map(hash_text)

n_unique_texts = units_df['text_hash'].nunique()
print(f'{len(units_df):,} text units → {n_unique_texts:,} unique text hashes')
if n_unique_texts < len(units_df):
    print(f'  ({len(units_df) - n_unique_texts} duplicate texts will share a cache entry)')

## 6. Provider-specific embedding functions

Each function takes a list of texts and returns a list of vectors. They handle batching, retries, and cost tracking internally. They never re-call the API for cached items.

In [ ]:
# Cost tracking globals
_voyage_cost = 0.0
_voyage_tokens = 0
_openai_cost = 0.0
_openai_tokens = 0

# Install dependencies
try:
    import voyageai
except ImportError:
    print('Installing voyageai...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'voyageai'])
    import voyageai

try:
    import openai
except ImportError:
    print('Installing openai...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openai'])
    import openai

_voyage_client = voyageai.Client(api_key=CharacterAI_Voyage)
_openai_client = openai.OpenAI(api_key=CharacterAI_OpenAI)

print(f'voyageai version: {voyageai.__version__ if hasattr(voyageai, "__version__") else "unknown"}')
print(f'openai version:   {openai.__version__}')

In [ ]:
def embed_voyage_batch(texts: List[str], input_type: str = 'document', max_retries: int = 3) -> Tuple[List[np.ndarray], int]:
    """Embed a batch of texts with Voyage. Returns (vectors, tokens_used).

    input_type: 'document' for indexed content, 'query' for queries. We use 'document' throughout
    since we're embedding for downstream similarity rather than retrieval.
    """
    for attempt in range(max_retries):
        try:
            result = _voyage_client.embed(
                texts=texts,
                model='voyage-3-large',
                input_type=input_type,
                output_dimension=1024,  # default; explicit for reproducibility
            )
            vectors = [np.array(v, dtype=np.float32) for v in result.embeddings]
            tokens = result.total_tokens
            return vectors, tokens
        except Exception as e:
            wait = 2 ** attempt
            print(f'  [Voyage retry {attempt+1}/{max_retries}] {type(e).__name__}: {e}; waiting {wait}s')
            time.sleep(wait)
    raise RuntimeError(f'Voyage failed after {max_retries} retries')

def embed_openai_batch(texts: List[str], max_retries: int = 3) -> Tuple[List[np.ndarray], int]:
    """Embed a batch with OpenAI text-embedding-3-large. Returns (vectors, tokens_used)."""
    for attempt in range(max_retries):
        try:
            result = _openai_client.embeddings.create(
                model='text-embedding-3-large',
                input=texts,
                encoding_format='float',
            )
            vectors = [np.array(d.embedding, dtype=np.float32) for d in result.data]
            tokens = result.usage.total_tokens
            return vectors, tokens
        except Exception as e:
            wait = 2 ** attempt
            print(f'  [OpenAI retry {attempt+1}/{max_retries}] {type(e).__name__}: {e}; waiting {wait}s')
            time.sleep(wait)
    raise RuntimeError(f'OpenAI failed after {max_retries} retries')

## 7. Pilot pass — embed first 10 items per model and verify

Before processing the full corpus, we embed a small sample with each model and check that vectors look right (correct dimensionality, non-zero, normalized to similar magnitudes).

In [ ]:
pilot_texts = units_df['text'].head(10).tolist()

print('=== PILOT: Voyage voyage-3-large ===')
voyage_pilot_vecs, voyage_pilot_tokens = embed_voyage_batch(pilot_texts)
print(f'  Returned {len(voyage_pilot_vecs)} vectors')
print(f'  Vector shape: {voyage_pilot_vecs[0].shape}')
print(f'  Vector dtype: {voyage_pilot_vecs[0].dtype}')
print(f'  Vector L2 norm (should be ~1 if normalized): {np.linalg.norm(voyage_pilot_vecs[0]):.4f}')
print(f'  Tokens used: {voyage_pilot_tokens}')
print(f'  Cost so far: ${voyage_pilot_tokens / 1e6 * VOYAGE_PRICE_PER_M:.4f}')

print('\n=== PILOT: OpenAI text-embedding-3-large ===')
openai_pilot_vecs, openai_pilot_tokens = embed_openai_batch(pilot_texts)
print(f'  Returned {len(openai_pilot_vecs)} vectors')
print(f'  Vector shape: {openai_pilot_vecs[0].shape}')
print(f'  Vector dtype: {openai_pilot_vecs[0].dtype}')
print(f'  Vector L2 norm (should be ~1 if normalized): {np.linalg.norm(openai_pilot_vecs[0]):.4f}')
print(f'  Tokens used: {openai_pilot_tokens}')
print(f'  Cost so far: ${openai_pilot_tokens / 1e6 * OPENAI_PRICE_PER_M:.4f}')

## 8. GO gate — confirm pilot looks right before full run

If the pilot above produced expected dimensions (1024 for Voyage, 3072 for OpenAI) and reasonable token counts, set `PROCEED = True` in the next cell to run the full embedding pass. If something looked off, leave it `False` and investigate.

In [ ]:
# Set to True after reviewing the pilot output above
PROCEED = True

if not PROCEED:
    raise RuntimeError(
        'Pilot review gate. Verify the pilot output above (vector shapes 1024 and 3072, '
        'L2 norms ~1, no errors). Then set PROCEED = True and re-run this cell.'
    )

print('GO confirmed. Proceeding with full embedding pass.')

## 9. Voyage full pass

Process all unique texts through Voyage, batching by 128, caching as we go. Already-cached items are skipped.

In [ ]:
def estimate_tokens(text: str) -> int:
    """Rough token estimate: chars / 3.5. Conservative (overestimates slightly)."""
    return max(1, int(len(text) / 3.5) + 1)

def make_token_aware_batches(
    df: pd.DataFrame,
    max_tokens_per_batch: int = 100_000,
    max_items_per_batch: int = 128,
) -> List[pd.DataFrame]:
    """Pack rows into batches that don't exceed max_tokens_per_batch.

    Each batch also capped at max_items_per_batch. Single texts larger than
    max_tokens_per_batch get their own batch and will probably fail (but we'd
    rather know than silently truncate).
    """
    batches = []
    current_rows = []
    current_tokens = 0

    for _, row in df.iterrows():
        text_tokens = estimate_tokens(row['text'])

        # Single oversized text gets its own batch
        if text_tokens >= max_tokens_per_batch:
            if current_rows:
                batches.append(pd.DataFrame(current_rows))
                current_rows = []
                current_tokens = 0
            batches.append(pd.DataFrame([row]))
            continue

        # Would adding this push us over either limit?
        if (current_tokens + text_tokens > max_tokens_per_batch) or (len(current_rows) >= max_items_per_batch):
            batches.append(pd.DataFrame(current_rows))
            current_rows = [row]
            current_tokens = text_tokens
        else:
            current_rows.append(row)
            current_tokens += text_tokens

    if current_rows:
        batches.append(pd.DataFrame(current_rows))

    return batches

def run_full_pass_api(
    units: pd.DataFrame,
    model_label: str,
    cache_subdir: Path,
    embed_fn,
    price_per_m: float,
    cost_ceiling: float,
    max_tokens_per_batch: int = 100_000,
) -> Tuple[Dict[str, np.ndarray], int, float]:
    """Process all unique texts through an API embedder. Returns (embeddings, tokens, cost)."""

    # Get unique (text, hash) pairs
    unique = units[['text', 'text_hash']].drop_duplicates(subset='text_hash').reset_index(drop=True)
    print(f'[{model_label}] {len(unique):,} unique texts to consider')

    # Filter out already-cached
    cached_hashes = {p.stem for p in cache_subdir.glob('*.npy')}
    to_embed = unique[~unique['text_hash'].isin(cached_hashes)].reset_index(drop=True)
    print(f'[{model_label}] {len(cached_hashes):,} already cached, {len(to_embed):,} to embed')

    cumulative_tokens = 0
    cumulative_cost = 0.0

    if len(to_embed) == 0:
        print(f'[{model_label}] All texts already cached.')
    else:
        # Token-aware batching — cap each batch at max_tokens_per_batch tokens
        batches = make_token_aware_batches(
            to_embed,
            max_tokens_per_batch=max_tokens_per_batch,
            max_items_per_batch=BATCH_SIZE_API,
        )
        n_batches = len(batches)
        total_est_tokens = sum(estimate_tokens(t) for t in to_embed['text'])
        print(f'[{model_label}] Packed into {n_batches} token-aware batches (~{total_est_tokens:,} tokens estimated)')

        t0 = time.time()
        for batch_i, batch in enumerate(batches):
            texts = batch['text'].tolist()
            hashes = batch['text_hash'].tolist()
            est_tok = sum(estimate_tokens(t) for t in texts)

            vecs, tokens = embed_fn(texts)
            cumulative_tokens += tokens
            cumulative_cost = cumulative_tokens / 1e6 * price_per_m

            # Save to cache
            for h, v in zip(hashes, vecs):
                cache_save(cache_subdir, h, v)

            elapsed = time.time() - t0
            rate = (batch_i + 1) / max(elapsed, 1e-9) * 60
            eta = (n_batches - batch_i - 1) / max(rate / 60, 1e-9)
            print(f'  Batch {batch_i+1:>3}/{n_batches}: {len(batch):>3} texts, '
                  f'est_tok={est_tok:>6}, actual_tok={tokens:>6}, cum_cost=${cumulative_cost:.4f}, '
                  f'rate={rate:.1f}/min, eta={eta:.0f}s')

            if cumulative_cost > cost_ceiling:
                raise RuntimeError(
                    f'[{model_label}] Cost ceiling ${cost_ceiling} exceeded '
                    f'(${cumulative_cost:.4f}). Aborting. Already-cached embeddings retained.'
                )

    # Load all cached vectors for the requested texts
    embeddings = {}
    for h in unique['text_hash']:
        v = cache_load(cache_subdir, h)
        if v is not None:
            embeddings[h] = v
        else:
            print(f'  [WARN] Missing cache for hash {h}')

    return embeddings, cumulative_tokens, cumulative_cost

voyage_emb, voyage_tokens, voyage_cost = run_full_pass_api(
    units_df, 'Voyage', CACHE_DIR / 'voyage', embed_voyage_batch,
    VOYAGE_PRICE_PER_M, COST_CEILING_VOYAGE,
    max_tokens_per_batch=100_000,  # Voyage limit is 120K; keep 20K headroom
)
print(f'\nVoyage: {len(voyage_emb):,} vectors loaded')
print(f'Voyage cost this run: ${voyage_cost:.4f} ({voyage_tokens:,} tokens)')


## 10. OpenAI full pass

In [ ]:
openai_emb, openai_tokens, openai_cost = run_full_pass_api(
    units_df, 'OpenAI', CACHE_DIR / 'openai', embed_openai_batch,
    OPENAI_PRICE_PER_M, COST_CEILING_OPENAI,
    max_tokens_per_batch=100_000,  # OpenAI limit is much higher; 100K is safe
)
print(f'\nOpenAI: {len(openai_emb):,} vectors loaded')
print(f'OpenAI cost this run: ${openai_cost:.4f} ({openai_tokens:,} tokens)')


## 11. Qwen3-Embedding-8B local pass

Loads the 8B-param Qwen3 embedding model in bfloat16 (~16GB VRAM) onto the Colab GPU. Embeds in batches of 8 to avoid OOM. Cached identically to the API embeddings.

First-run download is ~15GB and takes a few minutes; subsequent runs use the HuggingFace cache.

In [ ]:
# Install transformers if needed
try:
    import transformers
    from transformers import AutoTokenizer, AutoModel
except ImportError:
    print('Installing transformers...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                          'transformers>=4.51.0', 'accelerate', 'sentencepiece'])
    import transformers
    from transformers import AutoTokenizer, AutoModel

import torch
import torch.nn.functional as F

print(f'transformers version: {transformers.__version__}')
print(f'torch version:        {torch.__version__}')
print(f'CUDA available:       {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:                  {torch.cuda.get_device_name(0)}')

In [ ]:
QWEN_MODEL_NAME = 'Qwen/Qwen3-Embedding-8B'

print(f'Loading {QWEN_MODEL_NAME} (this may take a few minutes on first run)...')
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, padding_side='left')
qwen_model = AutoModel.from_pretrained(
    QWEN_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='cuda' if torch.cuda.is_available() else 'cpu',
)
qwen_model.eval()
print(f'Loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB' if torch.cuda.is_available() else 'Loaded on CPU.')
print(f'Embedding dimension: {qwen_model.config.hidden_size}')

In [ ]:
def last_token_pool(last_hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """Qwen3 embedding uses last-token pooling (per HuggingFace model card)."""
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

def embed_qwen_batch(texts: List[str], max_length: int = 8192) -> List[np.ndarray]:
    """Embed a batch of texts with Qwen3-Embedding-8B."""
    with torch.no_grad():
        encoded = qwen_tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt',
        ).to(qwen_model.device)

        outputs = qwen_model(**encoded)
        embeddings = last_token_pool(outputs.last_hidden_state, encoded['attention_mask'])
        embeddings = F.normalize(embeddings, p=2, dim=1)

    return [v.float().cpu().numpy() for v in embeddings]

# Pilot test with Qwen
print('Qwen3-8B pilot...')
qwen_pilot_vecs = embed_qwen_batch(pilot_texts[:3])
print(f'  Vector shape: {qwen_pilot_vecs[0].shape}')
print(f'  Vector L2 norm: {np.linalg.norm(qwen_pilot_vecs[0]):.4f}')
print(f'  Sample values: {qwen_pilot_vecs[0][:5]}')

In [ ]:
def run_full_pass_qwen(units: pd.DataFrame, cache_subdir: Path) -> Dict[str, np.ndarray]:
    """Process all unique texts through Qwen3-8B locally."""

    unique = units[['text', 'text_hash']].drop_duplicates(subset='text_hash').reset_index(drop=True)
    print(f'[Qwen3-8B] {len(unique):,} unique texts to consider')

    cached_hashes = {p.stem for p in cache_subdir.glob('*.npy')}
    to_embed = unique[~unique['text_hash'].isin(cached_hashes)].reset_index(drop=True)
    print(f'[Qwen3-8B] {len(cached_hashes):,} already cached, {len(to_embed):,} to embed')

    if len(to_embed) > 0:
        n_batches = (len(to_embed) + BATCH_SIZE_QWEN - 1) // BATCH_SIZE_QWEN
        t0 = time.time()

        for batch_i in range(n_batches):
            start = batch_i * BATCH_SIZE_QWEN
            end = min(start + BATCH_SIZE_QWEN, len(to_embed))
            batch = to_embed.iloc[start:end]

            texts = batch['text'].tolist()
            hashes = batch['text_hash'].tolist()

            vecs = embed_qwen_batch(texts)

            for h, v in zip(hashes, vecs):
                cache_save(cache_subdir, h, v)

            elapsed = time.time() - t0
            done = end
            eta_min = (len(to_embed) - done) / max(done / elapsed, 1e-9) / 60

            if (batch_i + 1) % 10 == 0 or batch_i == n_batches - 1:
                print(f'  Batch {batch_i+1:>4}/{n_batches}: {done}/{len(to_embed)} done, '
                      f'elapsed={elapsed/60:.1f}min, eta={eta_min:.1f}min')

    embeddings = {}
    for h in unique['text_hash']:
        v = cache_load(cache_subdir, h)
        if v is not None:
            embeddings[h] = v

    return embeddings

qwen_emb = run_full_pass_qwen(units_df, CACHE_DIR / 'qwen3')
print(f'\nQwen3-8B: {len(qwen_emb):,} vectors loaded')

# Free GPU memory
del qwen_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 12. all-mpnet-base-v2 local pass (preregistered baseline)

`sentence-transformers/all-mpnet-base-v2` was named in the preregistration (§10.1) as the embedding model for this study. We retain it as a preregistered baseline alongside the three current-generation upgrades. It is small (110M parameters, 768-dim), free, and runs locally in ~1 minute on the same GPU.

**Important property:** mpnet has a 384-token maximum sequence length. Texts longer than this (full conversations especially, which can exceed 5,000 tokens) will be truncated. This is a real-world limitation of the preregistered model and we document the truncation rate in the diagnostic at the end of this section.


In [ ]:
# Install sentence-transformers if needed
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    print('Installing sentence-transformers...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'])
    from sentence_transformers import SentenceTransformer

import sentence_transformers
print(f'sentence-transformers version: {sentence_transformers.__version__}')


In [ ]:
MPNET_MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'

print(f'Loading {MPNET_MODEL_NAME} (~420MB, downloads once)...')
mpnet_device = 'cuda' if torch.cuda.is_available() else 'cpu'
mpnet_model = SentenceTransformer(MPNET_MODEL_NAME, device=mpnet_device)

print(f'Loaded on {mpnet_device}.')
print(f'Embedding dimension: {mpnet_model.get_sentence_embedding_dimension()}')
print(f'Max sequence length: {mpnet_model.max_seq_length} tokens')
print(f'  (texts longer than this are truncated to the first {mpnet_model.max_seq_length} tokens)')


In [ ]:
def embed_mpnet_batch(texts: List[str]) -> List[np.ndarray]:
    """Embed a batch of texts with all-mpnet-base-v2.

    Sequences longer than the model max length (384 tokens) are truncated.
    Output is L2-normalized to match the other three embedding models.
    """
    vecs = mpnet_model.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return [v.astype(np.float32) for v in vecs]

# Pilot test
print('mpnet pilot...')
mpnet_pilot_vecs = embed_mpnet_batch(pilot_texts[:3])
print(f'  Vector shape: {mpnet_pilot_vecs[0].shape}')
print(f'  Vector L2 norm: {np.linalg.norm(mpnet_pilot_vecs[0]):.4f}')
print(f'  Sample values: {mpnet_pilot_vecs[0][:5]}')


In [ ]:
def run_full_pass_mpnet(units: pd.DataFrame, cache_subdir: Path) -> Dict[str, np.ndarray]:
    """Process all unique texts through all-mpnet-base-v2 locally."""

    unique = units[['text', 'text_hash']].drop_duplicates(subset='text_hash').reset_index(drop=True)
    print(f'[mpnet] {len(unique):,} unique texts to consider')

    cached_hashes = {p.stem for p in cache_subdir.glob('*.npy')}
    to_embed = unique[~unique['text_hash'].isin(cached_hashes)].reset_index(drop=True)
    print(f'[mpnet] {len(cached_hashes):,} already cached, {len(to_embed):,} to embed')

    # Track truncation rate (for the manifest and diagnostic)
    n_truncated = 0
    if len(to_embed) > 0:
        tokenizer = mpnet_model.tokenizer
        max_len = mpnet_model.max_seq_length
        for text in to_embed['text']:
            n_tok = len(tokenizer.encode(text, add_special_tokens=True))
            if n_tok > max_len:
                n_truncated += 1
        pct_trunc = 100 * n_truncated / len(to_embed)
        print(f'[mpnet] {n_truncated}/{len(to_embed)} texts ({pct_trunc:.1f}%) exceed {max_len} tokens and will be truncated')

        n_batches = (len(to_embed) + BATCH_SIZE_MPNET - 1) // BATCH_SIZE_MPNET
        t0 = time.time()

        for batch_i in range(n_batches):
            start = batch_i * BATCH_SIZE_MPNET
            end = min(start + BATCH_SIZE_MPNET, len(to_embed))
            batch = to_embed.iloc[start:end]

            texts = batch['text'].tolist()
            hashes = batch['text_hash'].tolist()

            vecs = embed_mpnet_batch(texts)

            for h, v in zip(hashes, vecs):
                cache_save(cache_subdir, h, v)

            if (batch_i + 1) % 20 == 0 or batch_i == n_batches - 1:
                elapsed = time.time() - t0
                done = end
                eta = (len(to_embed) - done) / max(done / elapsed, 1e-9)
                print(f'  Batch {batch_i+1:>3}/{n_batches}: {done}/{len(to_embed)} done, '
                      f'elapsed={elapsed:.1f}s, eta={eta:.1f}s')

    embeddings = {}
    for h in unique['text_hash']:
        v = cache_load(cache_subdir, h)
        if v is not None:
            embeddings[h] = v

    return embeddings, n_truncated

mpnet_emb, mpnet_n_truncated = run_full_pass_mpnet(units_df, CACHE_DIR / 'mpnet')
print(f'\nmpnet: {len(mpnet_emb):,} vectors loaded')

# Free model memory
del mpnet_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None


## 13. Assemble final per-model parquet files

Join the cached vectors back to `units_df` and write three parquet files (one per model). Each row has the unit metadata plus the embedding vector.

In [ ]:
def assemble_parquet(units: pd.DataFrame, embeddings: Dict[str, np.ndarray], model_label: str, out_path: Path) -> int:
    df = units.copy()
    df['embedding'] = df['text_hash'].map(embeddings)

    n_missing = df['embedding'].isna().sum()
    if n_missing > 0:
        print(f'[{model_label}] WARNING: {n_missing} rows have no embedding (cache miss). Dropping.')
        df = df.dropna(subset=['embedding'])

    # Convert embedding column to list-of-floats for parquet compatibility
    df['embedding'] = df['embedding'].map(lambda v: v.astype(np.float32).tolist())

    # Drop the raw text from the parquet to keep file size manageable
    # (text is recoverable from chat_turns_long.parquet / open_ends_long.parquet)
    df_out = df.drop(columns=['text'])

    df_out.to_parquet(out_path, index=False)
    size_mb = out_path.stat().st_size / 1024 / 1024
    print(f'[{model_label}] Wrote {out_path.name}: {len(df_out):,} rows, {size_mb:.1f} MB')
    return len(df_out)

n_voyage = assemble_parquet(units_df, voyage_emb, 'Voyage',   EMB_DIR / 'voyage_3_large.parquet')
n_openai = assemble_parquet(units_df, openai_emb, 'OpenAI',   EMB_DIR / 'openai_3_large.parquet')
n_qwen   = assemble_parquet(units_df, qwen_emb,   'Qwen3-8B', EMB_DIR / 'qwen3_8b.parquet')
n_mpnet  = assemble_parquet(units_df, mpnet_emb,  'mpnet',    EMB_DIR / 'all_mpnet_base_v2.parquet')


## 14. Sanity check — pairwise similarity across models

If all three models are working sensibly, then within a single model the cosine similarity between two random texts should be modest (~0.3-0.6 typical), and within-conversation similarities should generally be higher than cross-conversation similarities. We just verify directionally.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def cosine_diagnostic(parquet_path: Path, model_label: str):
    df = pd.read_parquet(parquet_path)

    # Get full conversation embeddings
    conv = df[df['unit_type'] == 'full_conversation'].copy()
    if len(conv) == 0:
        print(f'[{model_label}] No full_conversation embeddings to diagnose')
        return

    vecs = np.stack(conv['embedding'].values)
    sims = cosine_similarity(vecs)
    np.fill_diagonal(sims, np.nan)

    cond = conv['condition_label'].values
    same_cond = cond[:, None] == cond[None, :]
    np.fill_diagonal(same_cond, False)

    sim_same = sims[same_cond]
    sim_diff = sims[~same_cond & ~np.isnan(sims)]

    print(f'[{model_label}] Conversation-pair cosine similarities:')
    print(f'  All pairs:        mean={np.nanmean(sims):.3f}, sd={np.nanstd(sims):.3f}, '
          f'min={np.nanmin(sims):.3f}, max={np.nanmax(sims):.3f}')
    print(f'  Within condition: mean={sim_same.mean():.3f}, n={len(sim_same):,}')
    print(f'  Cross condition:  mean={sim_diff.mean():.3f}, n={len(sim_diff):,}')
    print(f'  Within - cross:   {sim_same.mean() - sim_diff.mean():+.4f}')

for label, path in [('Voyage',   EMB_DIR / 'voyage_3_large.parquet'),
                    ('OpenAI',   EMB_DIR / 'openai_3_large.parquet'),
                    ('Qwen3-8B', EMB_DIR / 'qwen3_8b.parquet'),
                    ('mpnet',    EMB_DIR / 'all_mpnet_base_v2.parquet')]:
    cosine_diagnostic(path, label)
    print()


## 15. Manifest and final summary

In [ ]:
manifest = {
    'generated_at': datetime.now().isoformat(timespec='seconds'),
    'models': {
        'voyage_3_large': {
            'provider': 'Voyage AI',
            'model_name': 'voyage-3-large',
            'dim': 1024,
            'n_vectors': n_voyage,
            'cost_this_run': round(voyage_cost, 4),
            'tokens_this_run': voyage_tokens,
            'role': 'primary (current-gen retrieval-optimized encoder)',
        },
        'openai_3_large': {
            'provider': 'OpenAI',
            'model_name': 'text-embedding-3-large',
            'dim': 3072,
            'n_vectors': n_openai,
            'cost_this_run': round(openai_cost, 4),
            'tokens_this_run': openai_tokens,
            'role': 'secondary (current-gen Matryoshka transformer)',
        },
        'qwen3_8b': {
            'provider': 'Alibaba (HuggingFace)',
            'model_name': 'Qwen/Qwen3-Embedding-8B',
            'dim': 4096,
            'n_vectors': n_qwen,
            'cost_this_run': 0.0,
            'tokens_this_run': None,
            'role': 'tertiary (open-weights, decoder-based with last-token pooling)',
        },
        'all_mpnet_base_v2': {
            'provider': 'sentence-transformers (HuggingFace)',
            'model_name': 'sentence-transformers/all-mpnet-base-v2',
            'dim': 768,
            'n_vectors': n_mpnet,
            'cost_this_run': 0.0,
            'tokens_this_run': None,
            'role': 'preregistered baseline (contrastive sentence-similarity, 384-token cap)',
            'n_truncated_this_run': int(mpnet_n_truncated),
        },
    },
    'units_embedded': units_df['unit_type'].value_counts().to_dict(),
    'total_unique_texts': int(units_df['text_hash'].nunique()),
    'total_text_units': int(len(units_df)),
}

manifest_path = EMB_DIR / 'embedding_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print('=' * 70)
print('NOTEBOOK 02 COMPLETE')
print('=' * 70)
print(f'Voyage:    {n_voyage:>5} vectors @ 1024-dim, ${voyage_cost:.4f}')
print(f'OpenAI:    {n_openai:>5} vectors @ 3072-dim, ${openai_cost:.4f}')
print(f'Qwen3-8B:  {n_qwen:>5} vectors @ 4096-dim, $0.00 (local)')
print(f'mpnet:     {n_mpnet:>5} vectors @  768-dim, $0.00 (local, {mpnet_n_truncated} truncated)')
print(f'\nTotal API spend this run: ${voyage_cost + openai_cost:.4f}')
print(f'\nOutputs:')
print(f'  {EMB_DIR / "voyage_3_large.parquet"}')
print(f'  {EMB_DIR / "openai_3_large.parquet"}')
print(f'  {EMB_DIR / "qwen3_8b.parquet"}')
print(f'  {EMB_DIR / "all_mpnet_base_v2.parquet"}')
print(f'  {manifest_path}')
print(f'\nCache retained at: {CACHE_DIR}')
print(f'  (re-running this notebook will read from cache; safe to keep.)')


## 16. Wrap-up checklist

Before moving to Notebook 03, confirm:

- [ ] All three models reported expected vector dimensions (1024, 3072, 4096)
- [ ] Cost stayed under ceilings (Voyage <$5, OpenAI <$5)
- [ ] Cosine diagnostic showed sensible values (within-condition mean > cross-condition mean ideally, or at minimum no extremely high overall similarity that would indicate degenerate embeddings)
- [ ] Three parquet files in `data/derived/embeddings/`
- [ ] `embedding_manifest.json` written

If everything looks good, Notebook 03 (open-coding rubric pass with Sonnet) can run. That's the one where Sonnet reads ~30 OEs and surfaces what reasoning frames actually appear, so we can lock the codebook before the human and LLM coding.